# Analyse des citations et échos de Deutéronome 6-8 dans Matthieu 4
Ce carnet Jupyter vise à identifier et analyser les relations intertextuelles (citations explicites, allusions, échos de mots-clés) entre Deutéronome 6-8 (LXX) et Matthieu 4.
Nous utiliserons le modèle de langage de traitement du langage naturel (NLP) pour le grec ancien, `odyCy`.

## Justification du traitement des mots vides (Stop Words)
Dans cette analyse, nous faisons l'hypothèse qu'il faut filtrer les mots vides (conjonctions `καί`, `δέ`, articles `ὁ`, prépositions très communes).
En grec ancien, conserver ces termes pour calculer la **densité de vocabulaire partagé** ou la **similarité sémantique** fausse les résultats, car ils créent un "bruit" important. Deux textes sans aucun rapport sémantique pourraient avoir un score de similarité élevé simplement parce qu'ils sont riches en conjonctions.
Cependant, pour la **recherche de citations exactes (N-Grams)**, leur retrait est parfois débattu, mais nous les enlèverons pour repérer les séquences de concepts "forts".


In [36]:
import spacy
import pandas as pd
import requests
import re
import warnings

# Ignorer certains avertissements liés à PyTorch/spaCy
warnings.filterwarnings("ignore", category=FutureWarning, module="thinc.shims.pytorch")

# Charger le modèle odyCy pour le grec ancien
print("Chargement du modèle odyCy...")
nlp = spacy.load("grc_odycy_joint_trf")
print("Modèle chargé !")


Chargement du modèle odyCy...
Modèle chargé !


## 1. Acquisition des données
Nous récupérons ici les textes de Matthieu 4 (GNT) et de Deutéronome 6-8 (LXX). 
Pour la simplicité de ce notebook, nous allons utiliser les données brutes extraites de projets open-source comme STEPBible et Eliran Wong's LXX (que nous simplifions ici en récupérant les textes bruts grecs).


In [37]:
# Textes bruts (simplifiés pour l'exemple, dans un projet réel, nous parserions les fichiers TSV ou JSON de STEPBible/Rahlfs)
# Pour une vraie implémentation, on pourrait utiliser des requêtes API ou lire des CSV locaux.
# Ici, nous mettons les textes grecs correspondants (version non accentuée ou accentuée)

# Deutéronome 6-8 (Extraits représentatifs ou texte complet)
# (Pour le notebook, nous simulons le texte avec le début de Deut 6 et Deut 8 pour la démonstration)
deut_text = """
καὶ αὗται αἱ ἐντολαὶ καὶ τὰ δικαιώματα καὶ τὰ κρίματα ὅσα ἐνετείλατο κύριος ὁ θεὸς ἡμῶν διδάξαι ὑμᾶς ποιεῖν οὕτως ἐν τῇ γῇ εἰς ἣν ὑμεῖς εἰσπορεύεσθε ἐκεῖ κληρονομῆσαι αὐτήν
ἵνα φοβῆσθε κύριον τὸν θεὸν ὑμῶν φυλάσσεσθαι πάντα τὰ δικαιώματα αὐτοῦ καὶ τὰς ἐντολὰς αὐτοῦ ὅσας ἐγὼ ἐντέλλομαί σοι σήμερον σὺ καὶ οἱ υἱοί σου καὶ οἱ υἱοὶ τῶν υἱῶν σου πάσας τὰς ἡμέρας τῆς ζωῆς σου ἵνα μακροημερεύσητε
καὶ ἄκουσον ισραηλ καὶ φύλαξαι ποιεῖν ὅπως εὖ σοι γένηται καὶ ἵνα πληθυνθῆτε σφόδρα καθάπερ ἐλάλησεν κύριος ὁ θεὸς τῶν πατέρων σου δοῦναί σοι γῆν ῥέουσαν γάλα καὶ μέλι
ἄκουε ισραηλ κύριος ὁ θεὸς ἡμῶν κύριος εἷς ἐστιν
καὶ ἀγαπήσεις κύριον τὸν θεόν σου ἐξ ὅλης τῆς καρδίας σου καὶ ἐξ ὅλης τῆς ψυχῆς σου καὶ ἐξ ὅλης τῆς δυνάμεώς σου
καὶ ἔσται τὰ ῥήματα ταῦτα ὅσα ἐγὼ ἐντέλλομαί σοι σήμερον ἐν τῇ καρδίᾳ σου καὶ ἐν τῇ ψυχῇ σου
μνησθήσῃ πάσης τῆς ὁδοῦ ἧς ἤγαγέν σε κύριος ὁ θεός σου ἐν τῇ ἐρήμῳ
οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι τῷ ἐκπορευομένῳ διὰ στόματος θεοῦ
κύριον τὸν θεόν σου φοβηθήσῃ καὶ αὐτῷ μόνῳ λατρεύσεις
οὐκ ἐκπειράσεις κύριον τὸν θεόν σου
""" # Extraits de Deut 6 et 8 (LXX) incluant le Shema et les réponses de Jésus

# Matthieu 4 (GNT) - La Tentation au désert
mt4_text = """
Τότε ὁ Ἰησοῦς ἀνήχθη εἰς τὴν ἔρημον ὑπὸ τοῦ πνεύματος πειρασθῆναι ὑπὸ τοῦ διαβόλου.
καὶ νηστεύσας ἡμέρας τεσσεράκοντα καὶ νύκτας τεσσεράκοντα, ὕστερον ἐπείνασεν.
καὶ προσελθὼν ὁ πειράζων εἶπεν αὐτῷ, Εἰ υἱὸς εἶ τοῦ θεοῦ, εἰπὲ ἵνα οἱ λίθοι οὗτοι ἄρτοι γένωνται.
ὁ δὲ ἀποκριθεὶς εἶπεν, Γέγραπται, Οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος, ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ.
Τότε παραλαμβάνει αὐτὸν ὁ διάβολος εἰς τὴν ἁγίαν πόλιν, καὶ ἔστησεν αὐτὸν ἐπὶ τὸ πτερύγιον τοῦ ἱεροῦ,
καὶ λέγει αὐτῷ, Εἰ υἱὸς εἶ τοῦ θεοῦ, βάλε σεαυτὸν κάτω· γέγραπται γὰρ ὅτι Τοῖς ἀγγέλοις αὐτοῦ ἐντελεῖται περὶ σοῦ καὶ ἐπὶ χειρῶν ἀροῦσίν σε, μήποτε προσκόψῃς πρὸς λίθον τὸν πόδα σου.
ἔφη αὐτῷ ὁ Ἰησοῦς, Πάλιν γέγραπται, Οὐκ ἐκπειράσεις κύριον τὸν θεόν σου.
Πάλιν παραλαμβάνει αὐτὸν ὁ διάβολος εἰς ὄρος ὑψηλὸν λίαν, καὶ δείκνυσιν αὐτῷ πάσας τὰς βασιλείας τοῦ κόσμου καὶ τὴν δόξαν αὐτῶν,
καὶ εἶπεν αὐτῷ, Ταῦτά σοι πάντα δώσω, ἐὰν πεσὼν προσκυνήσῃς μοι.
τότε λέγει αὐτῷ ὁ Ἰησοῦς, Ὕπαγε, Σατανᾶ· γέγραπται γάρ, Κύριον τὸν θεόν σου προσκυνήσεις καὶ αὐτῷ μόνῳ λατρεύσεις.
"""


## 2. Prétraitement et Lemmatisation
La fonction suivante tokenise le texte, retire la ponctuation et les espaces, et surtout **filtre les mots vides (stop words)**.


In [38]:
# Liste basique de mots vides en grec (articles, conjonctions courantes)
STOP_WORDS = {"ὁ", "ἡ", "τό", "οἱ", "αἱ", "τά", "καί", "δέ", "τε", "μέν", "γάρ", "ἐν", "εἰς", "πρός", "ἐπί", "ἀπό", "ἐκ", "ὑπό", "σύν", "ὅτι", "εἰ", "ἵνα", "αὐτός", "σύ", "ἐγώ", "οὗτος", "ἐκεῖνος", "ὅς", "τὶς", "εἰμί", "οὐ", "οὐκ", "μή"}

def preprocess(text, remove_stopwords=True):
    # Traitement avec le modèle spaCy
    doc = nlp(text.lower())
    lemmas = []
    for token in doc:
        # Ignorer ponctuation et espaces
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_
        if remove_stopwords and lemma in STOP_WORDS:
            continue
        lemmas.append(lemma)
    return lemmas

# Extraire les lemmes (avec et sans stop words pour comparer si besoin)
lemmes_deut = preprocess(deut_text)
lemmes_mt = preprocess(mt4_text)

print(f"Lemmes de Deutéronome extraits : {len(lemmes_deut)}")
print(f"Lemmes de Matthieu 4 extraits : {len(lemmes_mt)}")


Lemmes de Deutéronome extraits : 102
Lemmes de Matthieu 4 extraits : 96


## 3. Méthode A : N-Grams de Lemmes (Citations quasi-littérales)
On cherche des séquences exactes de *n* lemmes qui se suivent dans les deux textes.


In [39]:
def find_ngrams(lemmas_source, lemmas_target, n=3):
    # Construire les n-grams pour la source
    source_ngrams = set([" ".join(lemmas_source[i:i+n]) for i in range(len(lemmas_source)-n+1)])
    
    # Trouver les correspondances dans la cible
    matches = []
    for i in range(len(lemmas_target)-n+1):
        target_ngram = " ".join(lemmas_target[i:i+n])
        if target_ngram in source_ngrams:
            matches.append({
                "position_mt": i,
                "ngram_match": target_ngram
            })
            
    return pd.DataFrame(matches)

# Recherche de trigrammes (3 lemmes consécutifs)
df_ngrams = find_ngrams(lemmes_deut, lemmes_mt, n=3)
print("Citations quasi-littérales (Trigrammes) trouvées :")
display(df_ngrams)


Citations quasi-littérales (Trigrammes) trouvées :


,position_mt,ngram_match
0,26,ἄρτος μόνος ζῶ
1,27,μόνος ζῶ ἄνθρωπος
2,28,ζῶ ἄνθρωπος ἀλλά
3,29,ἄνθρωπος ἀλλά πᾶς
4,30,ἀλλά πᾶς ῥῆμα
5,31,πᾶς ῥῆμα ἐκπορεύομαι
6,32,ῥῆμα ἐκπορεύομαι διά
7,33,ἐκπορεύομαι διά στόμα
8,34,διά στόμα θεός
9,65,ἐκπειράζω κύριος θεός


## 4. Méthode B : Densité de Vocabulaire Partagé (Fenêtre glissante)
On utilise une fenêtre glissante sur le texte de Matthieu 4 pour repérer des concentrations de vocabulaire typique de Deutéronome.


In [40]:
from IPython.display import HTML, display

def find_echoes_density(source_lemmas, target_text, window_size=15, threshold=3):
    doc_target = nlp(target_text)
    # Ne garder que les tokens pertinents pour la fenêtre (hors ponctuation)
    valid_tokens = [t for t in doc_target if not t.is_punct and not t.is_space]
    
    echoes = []
    source_set = set(source_lemmas)
    
    for i in range(len(valid_tokens) - window_size):
        window = valid_tokens[i : i + window_size]
        # Lemmes de la fenêtre, sans les stop words
        window_lemmas = [t.lemma_ for t in window if t.lemma_ not in STOP_WORDS]
        
        matches = set(window_lemmas).intersection(source_set)
        
        if len(matches) >= threshold:
            # Créer le texte avec les mots correspondants en GRAS et ROUGE
            highlighted_text = []
            for t in window:
                if t.lemma_ in matches:
                    highlighted_text.append(f'<b style="color: #d9534f;">{t.text}</b>')
                else:
                    highlighted_text.append(t.text)
                    
            echoes.append({
                "Position": i,
                "Texte_Fenêtre (Mt 4)": " ".join(highlighted_text),
                "Lemmes_Correspondants": ", ".join(matches),
                "Nombre_Correspondances": len(matches),
                "Score_Densité": round(len(matches) / window_size, 2)
            })
            
    return pd.DataFrame(echoes)

df_echoes = find_echoes_density(lemmes_deut, mt4_text, window_size=12, threshold=3)
print("Zones de haute densité (Échos) avec correspondances en rouge et gras :")
if not df_echoes.empty:
    df_echoes = df_echoes.sort_values(by="Score_Densité", ascending=False).head(10)
    display(HTML(df_echoes.to_html(escape=False, index=False)))
else:
    print("Aucune densité n'atteint le seuil.")


Zones de haute densité (Échos) avec correspondances en rouge et gras :


Position,Texte_Fenêtre (Mt 4),Lemmes_Correspondants,Nombre_Correspondances,Score_Densité
49,μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ,"στόμα, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, μόνος, ἄνθρωπος, ζῶ, πᾶς, θεός",10,0.83
48,ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος,"ἄρτος, διά, στόμα, ἐκπορεύομαι, ἀλλά, ῥῆμα, μόνος, ἄνθρωπος, ζῶ, πᾶς",10,0.83
50,ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ Τότε,"στόμα, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, ἄνθρωπος, ζῶ, πᾶς, θεός",9,0.75
47,ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ,"ἄρτος, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, μόνος, ἄνθρωπος, ζῶ, πᾶς",9,0.75
52,ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ Τότε παραλαμβάνει αὐτὸν,"στόμα, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, ἄνθρωπος, πᾶς, θεός",8,0.67
51,ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ Τότε παραλαμβάνει,"στόμα, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, ἄνθρωπος, πᾶς, θεός",8,0.67
46,Οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ,"ἄρτος, ἐκπορεύομαι, ἀλλά, ῥῆμα, μόνος, ἄνθρωπος, ζῶ, πᾶς",8,0.67
45,Γέγραπται Οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι,"ἄρτος, ῥῆμα, ἀλλά, μόνος, ἄνθρωπος, ζῶ, πᾶς",7,0.58
53,ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ Τότε παραλαμβάνει αὐτὸν ὁ,"στόμα, διά, ἐκπορεύομαι, ἀλλά, ῥῆμα, πᾶς, θεός",7,0.58
54,ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ Τότε παραλαμβάνει αὐτὸν ὁ διάβολος,"στόμα, διά, ἐκπορεύομαι, ῥῆμα, πᾶς, θεός",6,0.50


## 5. Méthode C : Analyse Sémantique (Embeddings)
Grâce au modèle Transformer `odyCy`, chaque token ou span possède une représentation vectorielle contextuelle. On peut comparer la similarité cosinus de phrases ou de groupes de mots pour capter des allusions thématiques, même quand les mots sont différents.


In [41]:
import numpy as np
from numpy.linalg import norm
from IPython.display import HTML, display

# Fonction de similarité cosinus manuelle utilisant les tenseurs (embeddings) du transformer odyCy
def get_sentence_embedding(sent):
    try:
        sent_doc = nlp(sent.text)
        if hasattr(sent_doc._, 'trf_data') and sent_doc._.trf_data is not None:
            emb = sent_doc._.trf_data.tensors[-1].flatten()
            if emb.shape[0] > 0:
                return emb
    except Exception as e:
        pass
    return None

def cosine_similarity(v1, v2):
    if v1 is None or v2 is None: return 0.0
    v1_norm, v2_norm = norm(v1), norm(v2)
    if v1_norm == 0 or v2_norm == 0: return 0.0
    return np.dot(v1, v2) / (v1_norm * v2_norm)

# Extraire les phrases de Mt 4 et Deut
mt_sentences = [sent for sent in nlp(mt4_text).sents if len(sent.text.strip()) > 5]
deut_sentences = [sent for sent in nlp(deut_text).sents if len(sent.text.strip()) > 5]

print(f"Comparaison de {len(mt_sentences)} phrases de Mt 4 avec {len(deut_sentences)} phrases de Deut 6-8...")

semantic_matches = []

for mt_sent in mt_sentences:
    mt_emb = get_sentence_embedding(mt_sent)
    if mt_emb is None: continue
    
    best_score = 0
    best_deut_sent = None
    
    for deut_sent in deut_sentences:
        deut_emb = get_sentence_embedding(deut_sent)
        if deut_emb is None: continue
        
        sim = cosine_similarity(mt_emb, deut_emb)
        if sim > best_score:
            best_score = sim
            best_deut_sent = deut_sent
            
    if best_score > 0.64: # Seuil abaissé (autour de 0.65)
        
        # Mettre en évidence les lemmes communs (hors mots vides)
        mt_lemmas = {t.lemma_ for t in mt_sent if t.lemma_ not in STOP_WORDS and not t.is_punct}
        deut_lemmas = {t.lemma_ for t in best_deut_sent if t.lemma_ not in STOP_WORDS and not t.is_punct}
        common_lemmas = mt_lemmas.intersection(deut_lemmas)
        
        def highlight_common(sent_doc, common):
            highlighted = []
            for t in sent_doc:
                if t.lemma_ in common:
                    # Rouge et Gras pour les mots correspondants
                    highlighted.append(f'<b style="color: #d9534f;">{t.text}</b>')
                else:
                    highlighted.append(t.text)
            return " ".join(highlighted)

        semantic_matches.append({
            "Mt_4_Phrase": highlight_common(mt_sent, common_lemmas),
            "Deut_Phrase_Similaire": highlight_common(best_deut_sent, common_lemmas),
            "Score_Similarité": round(float(best_score), 3)
        })

df_semantic = pd.DataFrame(semantic_matches)
print("Allusions thématiques / sémantiques identifiées (mots communs en gras et couleur) :")
if not df_semantic.empty:
    df_semantic = df_semantic.sort_values(by="Score_Similarité", ascending=False).head(10)
    # Affichage du dataframe en interprétant le HTML pour voir la couleur
    display(HTML(df_semantic.to_html(escape=False, index=False)))
else:
    print("Aucune allusion sémantique dépassant le seuil n'a été trouvée.")


Comparaison de 12 phrases de Mt 4 avec 9 phrases de Deut 6-8...
Allusions thématiques / sémantiques identifiées (mots communs en gras et couleur) :


Mt_4_Phrase,Deut_Phrase_Similaire,Score_Similarité
καὶ προσελθὼν ὁ πειράζων εἶπεν αὐτῷ,κύριον τὸν θεόν σου φοβηθήσῃ καὶ αὐτῷ μόνῳ λατρεύσεις,0.752
ὁ δὲ ἀποκριθεὶς εἶπεν,κύριον τὸν θεόν σου φοβηθήσῃ καὶ αὐτῷ μόνῳ λατρεύσεις,0.744
"Πάλιν παραλαμβάνει αὐτὸν ὁ διάβολος εἰς ὄρος ὑψηλὸν λίαν , καὶ δείκνυσιν αὐτῷ πάσας τὰς βασιλείας τοῦ κόσμου καὶ τὴν δόξαν αὐτῶν , \n καὶ εἶπεν αὐτῷ , Ταῦτά σοι πάντα δώσω , ἐὰν πεσὼν προσκυνήσῃς μοι .",οὐκ ἐκπειράσεις κύριον τὸν θεόν σου,0.728
"γέγραπται γὰρ ὅτι Τοῖς ἀγγέλοις αὐτοῦ ἐντελεῖται περὶ σοῦ καὶ ἐπὶ χειρῶν ἀροῦσίν σε , μήποτε προσκόψῃς πρὸς λίθον τὸν πόδα σου .",οὐκ ἐκπειράσεις κύριον τὸν θεόν σου,0.724
Τότε ὁ Ἰησοῦς ἀνήχθη εἰς τὴν ἔρημον ὑπὸ τοῦ πνεύματος πειρασθῆναι ὑπὸ τοῦ διαβόλου .,οὐκ ἐκπειράσεις κύριον τὸν θεόν σου,0.708
", Γέγραπται , Οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος , ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ .",οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος ἀλλ' ἐπὶ παντὶ ῥήματι τῷ ἐκπορευομένῳ διὰ στόματος θεοῦ \n,0.701
"γέγραπται γάρ , Κύριον τὸν θεόν σου προσκυνήσεις καὶ αὐτῷ μόνῳ λατρεύσεις .",κύριον τὸν θεόν σου φοβηθήσῃ καὶ αὐτῷ μόνῳ λατρεύσεις,0.697
"ἔφη αὐτῷ ὁ Ἰησοῦς , Πάλιν γέγραπται , Οὐκ ἐκπειράσεις κύριον τὸν θεόν σου .",οὐκ ἐκπειράσεις κύριον τὸν θεόν σου,0.687
"Τότε παραλαμβάνει αὐτὸν ὁ διάβολος εἰς τὴν ἁγίαν πόλιν , καὶ ἔστησεν αὐτὸν ἐπὶ τὸ πτερύγιον τοῦ ἱεροῦ , \n καὶ λέγει αὐτῷ , Εἰ υἱὸς εἶ τοῦ θεοῦ , βάλε σεαυτὸν κάτω ·",οὐκ ἐκπειράσεις κύριον τὸν θεόν σου,0.669


## 6. Affichage Interlinéaire (Grec - Glossaire Anglais)
Pour mieux comprendre les correspondances, il est souvent très utile de visualiser le texte grec avec une traduction interlinéaire mot à mot.
Les projets comme **STEPBible-Data** fournissent des lexiques complets (comme le TBESG - Tyndale Brief English Semantic Gloss) associant chaque lemme à une définition.
Ici, nous allons créer une fonction d'affichage interlinéaire en utilisant un dictionnaire simplifié (qui simule le chargement d'un fichier TSV de lexique).

In [42]:
# Simulation d'un mini-lexique basé sur STEPBible (Lemme -> Glossaire Anglais)
# Dans un projet complet, vous chargeriez ici un fichier comme 'TBESG.tsv' de STEPBible-Data
mini_lexicon = {
    "ὁ": "the", "δέ": "but", "ἀποκρίνομαι": "answer", "λέγω": "say/speak", 
    "γράφω": "write", "οὐ": "not", "ἐπί": "on/upon", "ἄρτος": "bread", 
    "μόνος": "alone/only", "ζάω": "live", "ἄνθρωπος": "man/human", 
    "ἀλλά": "but", "πᾶς": "all/every", "ῥῆμα": "word/spoken matter", 
    "ἐκπορεύομαι": "go out/proceed", "διά": "through", "στόμα": "mouth", 
    "θεός": "God", "καί": "and", "μνημονεύω": "remember", "πᾶς": "all",
    "ὁδός": "way", "ἄγω": "lead", "σύ": "you", "κύριος": "Lord",
    "ἐν": "in", "ἔρημος": "wilderness", "εἰμί": "be", "γίγνομαι": "become",
    "οὗτος": "this"
}

def print_interlinear(text, nlp_model, lexicon):
    doc = nlp_model(text)
    
    # Affichage formaté en deux lignes pour chaque groupe de mots
    # On va faire des blocs de 8 mots pour que ça tienne sur l'écran
    words_per_line = 8
    
    for i in range(0, len(doc), words_per_line):
        chunk = doc[i:i+words_per_line]
        
        # Ligne 1 : Texte Grec
        line1 = " ".join([f"{token.text:<15}" for token in chunk if not token.is_space])
        
        # Ligne 2 : Lemme ou Traduction
        glosses = []
        for token in chunk:
            if token.is_space: continue
            if token.is_punct:
                glosses.append(f"{token.text:<15}")
            else:
                # Chercher dans le lexique, sinon afficher le lemme ou "?"
                gloss = lexicon.get(token.lemma_, f"[{token.lemma_}]")
                glosses.append(f"{gloss:<15}")
                
        line2 = " ".join(glosses)
        
        print(line1)
        print(line2)
        print("-" * 80)

# Exemple sur l'une des citations trouvées : Matthieu 4:4
mt4_4 = "ὁ δὲ ἀποκριθεὶς εἶπεν, Γέγραπται, Οὐκ ἐπ' ἄρτῳ μόνῳ ζήσεται ὁ ἄνθρωπος, ἀλλ' ἐπὶ παντὶ ῥήματι ἐκπορευομένῳ διὰ στόματος θεοῦ."

print("Affichage Interlinéaire de Matthieu 4:4 :\n")
print_interlinear(mt4_4, nlp, mini_lexicon)


Affichage Interlinéaire de Matthieu 4:4 :

ὁ               δὲ              ἀποκριθεὶς      εἶπεν           ,               Γέγραπται       ,               Οὐκ            
the             but             [ἀποκρίνω]      say/speak       ,               write           ,               not            
--------------------------------------------------------------------------------
ἐπ'             ἄρτῳ            μόνῳ            ζήσεται         ὁ               ἄνθρωπος        ,               ἀλλ'           
on/upon         bread           alone/only      [ζῶ]            the             man/human       ,               but            
--------------------------------------------------------------------------------
ἐπὶ             παντὶ           ῥήματι          ἐκπορευομένῳ    διὰ             στόματος        θεοῦ            .              
on/upon         all             word/spoken matter go out/proceed  through         mouth           God             .              
------------------------